# 00 セットアップ

ハンズオン1（Genie Agent で、品質保証部の分析エージェントを作る）で使うデータを準備します。
**Genie Agent の作成と、テーブル・Volume の追加は手動で行うため、このノートブックでは行いません**（手順は `01_シナリオと想定QA` を参照）。

| 手順 | このノートブックで行うこと |
|---|---|
| 1 | スキーマと Volume（`docs`・`raw_data`）を作る |
| 2 | この Git フォルダの `csv/` と `unstructured_data/` を、Volume にコピーする |
| 3 | Volume にコピーした CSV を読み込み、列コメント付きのテーブルを4つ作る |
| 4 | 主キー・外部キー（情報提供用の制約）を付け、件数を確認する |

**準備するもの**
- サーバーレスのコンピュート（右上の［接続］から選ぶ）
- 指定するカタログに対する `USE CATALOG` と、スキーマを作る権限（`CREATE SCHEMA`）。既存のスキーマを使う場合は、そのスキーマへの `USE SCHEMA`・`CREATE TABLE`・`CREATE VOLUME`

上のウィジェットで **カタログ名** と **スキーマ名** を指定してから、［すべてを実行］を押してください（約2〜3分）。

In [0]:
dbutils.widgets.text("catalog", "fevm_jp_hunter_serverless_catalog", "カタログ名")
dbutils.widgets.text("schema", "bricks_electronics", "スキーマ名")

In [0]:
import os

# ウィジェットの値を取得
catalog = dbutils.widgets.get("catalog").strip()
schema_name = dbutils.widgets.get("schema").strip()
assert catalog, "カタログ名を指定してください"
assert schema_name, "スキーマ名を指定してください"

DOCS_VOLUME = "docs"      # Genie Agent に追加する文書（PDF・PPTX）
RAW_VOLUME = "raw_data"   # テーブルの元データ（CSV）。Genie Agent には追加しない

# このノートブックがある Git フォルダの場所（csv/ と unstructured_data/ はこの下にある）
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_dir = "/Workspace" + os.path.dirname(notebook_path)
src_csv_dir = f"{repo_dir}/csv"
src_docs_dir = f"{repo_dir}/unstructured_data"

docs_path = f"/Volumes/{catalog}/{schema_name}/{DOCS_VOLUME}"
raw_csv_path = f"/Volumes/{catalog}/{schema_name}/{RAW_VOLUME}/csv"

print(f"カタログ.スキーマ : {catalog}.{schema_name}")
print(f"コピー元（CSV）  : {src_csv_dir}")
print(f"コピー元（文書）  : {src_docs_dir}")
print(f"コピー先（CSV）  : {raw_csv_path}")
print(f"コピー先（文書）  : {docs_path}")

## 1. スキーマと Volume の作成

- `docs`: 取扱説明書・品質会議の資料・サービス技術情報・保証規定（PDF・PPTX）。**この Volume を Genie Agent に追加します**
- `raw_data`: テーブルの元データ（CSV）。Genie Agent には追加しません（関係のないファイルを入れないことが、文書の検索の精度につながるため）

Genie Agent の文書の分析（コンテンツ検索）は、**Unity Catalog のマネージド Volume** だけに対応しています。

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema_name}` "
          "COMMENT 'ブリックス電機（架空の家電メーカー）の修理受付ハンズオン用のデータ'")

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema_name}`.`{DOCS_VOLUME}`
COMMENT 'ブリックス電機の社内文書（PDF・PPTX）。manuals/ = 取扱説明書（製品ごとに1本）、quality_meetings/ = 品質保証部の月次の品質会議の資料、service_bulletins/ = サービス技術情報（既知の不具合と対処の通達）、policies/ = 保証規定・修理料金表'
""")
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema_name}`.`{RAW_VOLUME}`
COMMENT 'セットアップ用の元データ（CSV）。Genie Agent には追加しない'
""")
print(f"✅ スキーマ: {catalog}.{schema_name}")
print(f"✅ Volume  : {catalog}.{schema_name}.{DOCS_VOLUME} / {RAW_VOLUME}")

## 2. Git フォルダから Volume へファイルをコピー

`dbutils.fs.cp` の `recurse=True` で、フォルダごとコピーします。Git フォルダ（ワークスペースのファイル）を指すときは `file:/Workspace/...` と書きます。

In [0]:
dbutils.fs.cp(f"file:{src_csv_dir}", raw_csv_path, recurse=True)

for f in dbutils.fs.ls(raw_csv_path):
    print(f"✅ {f.name:<22} {f.size:>12,} bytes")

In [0]:
dbutils.fs.cp(f"file:{src_docs_dir}", docs_path, recurse=True)

total = 0
for d in dbutils.fs.ls(docs_path):
    files = dbutils.fs.ls(d.path)
    total += len(files)
    print(f"📁 {d.name:<20} {len(files):>3} ファイル")
    for f in files:
        print(f"     {f.name}")
print(f"\n🎉 文書 {total} ファイルを {docs_path} にコピーしました")

## 3. CSV を読み込んで、テーブルを作成

Volume の CSV を Spark で読み込み、テーブルにします。**列コメント**は Genie Agent が列の意味を理解する手がかりになるため、すべての列に付けます。

| テーブル | 内容 | 行数 |
|---|---|---|
| `products` | 製品（型番・カテゴリ・保証期間） | 10 |
| `production_lots` | 製造ロット（工場・製造月・製造台数） | 783 |
| `sales_monthly` | 月次の販売台数（型番×地域） | 1,080 |
| `repair_requests` | 修理受付（症状・対応結果・対応メモ） | 11,884 |

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType, TimestampType


def col(name, dtype, comment):
    return StructField(name, dtype, True, metadata={"comment": comment})


TABLES = {
    "products": dict(
        comment="製品マスタ。ブリックス電機（架空）が製造・販売する生活家電10製品",
        schema=StructType([
            col("model_no", StringType(), "型番（例: RC-500）。製品を一意に表すキー"),
            col("product_name", StringType(), "製品名（例: IH炊飯器、圧力IH炊飯器）"),
            col("category", StringType(), "製品カテゴリ（炊飯器・電気ケトル・電子レンジ・掃除機・空気清浄機・扇風機・暖房器具・加湿器・トースター）"),
            col("capacity", StringType(), "容量・サイズ（例: 5.5合、1.0L、25畳）。ない製品は空"),
            col("warranty_months", IntegerType(), "保証期間（か月）。お買い上げ日から数える。RC-550 だけ24か月、ほかは12か月"),
            col("launch_date", DateType(), "発売日"),
            col("list_price_jpy", IntegerType(), "希望小売価格（円・税込）"),
        ])),
    "production_lots": dict(
        comment="製造ロット。2024年4月〜2026年9月に製造したロット。修理受付の lot_no とつながる",
        schema=StructType([
            col("lot_no", StringType(), "ロット番号。「型番の略号-製造年月(YYMM)+ロット記号」（例: RC5-2604B = RC-500 を2026年4月に製造したBロット）。製造番号の先頭9文字と同じ"),
            col("model_no", StringType(), "型番（products.model_no）"),
            col("factory", StringType(), "製造した工場（新潟工場・タイ工場・蘇州工場）"),
            col("production_month", DateType(), "製造月（月の1日）"),
            col("quantity", IntegerType(), "製造台数"),
        ])),
    "sales_monthly": dict(
        comment="月次の販売実績（型番×地域）。2025年4月〜2026年9月。受付率（販売1,000台あたりの受付件数）の分母に使う",
        schema=StructType([
            col("sales_month", DateType(), "販売月（月の1日）"),
            col("model_no", StringType(), "型番（products.model_no）"),
            col("region", StringType(), "地域（北海道・東北／関東／中部／近畿／中国・四国／九州・沖縄）"),
            col("units_sold", IntegerType(), "販売台数（家電量販店とECの合計）"),
            col("sales_amount_jpy", LongType(), "販売金額（円）"),
        ])),
    "repair_requests": dict(
        comment="修理受付。2025年4月〜2026年9月に、コールセンター（電話・Web）で受け付けた修理の依頼。1行が1件の受付",
        schema=StructType([
            col("request_id", StringType(), "受付ID（R-受付年月(YYMM)-受付番号）。受付番号は修理以外の問い合わせと共通の連番のため、飛び番がある"),
            col("received_at", TimestampType(), "受付日時"),
            col("channel", StringType(), "受付チャネル（電話・Web）"),
            col("model_no", StringType(), "型番（products.model_no）"),
            col("serial_no", StringType(), "製造番号（ロット番号-5桁の連番。例: RC5-2604B-00871）"),
            col("lot_no", StringType(), "ロット番号（production_lots.lot_no）"),
            col("purchase_date", DateType(), "お買い上げ日"),
            col("region", StringType(), "お客様の地域（sales_monthly.region と同じ区分）"),
            col("symptom_category", StringType(), "症状の区分（電源が入らない・途中で止まる・異音・におい・エラー表示・水漏れ・性能低下・破損）"),
            col("result_category", StringType(), "対応結果。持込修理=引き取って修理／出張修理=訪問して修理／製品交換=初期不良などで新品と交換／修理不要=電話・メールで使いかたやお手入れを案内して解決／異常なし=修理を手配したが、点検の結果、故障ではなかった"),
            col("warranty_status", StringType(), "保証の状態（保証内・保証外）。受付日がお買い上げ日から保証期間内かどうか"),
            col("service_cost_jpy", IntegerType(), "当社負担の費用（円）。保証期間内の修理・交換・点検の技術料・部品代・送料。保証期間外（お客様負担）や、案内だけで解決した場合は0"),
            col("operator_memo", StringType(), "オペレーターの対応メモ。お客様の氏名・電話番号・メールアドレス・住所（個人情報）を含む。回答や集計結果に個人情報を出さないこと"),
        ])),
}

# 再実行できるよう、外部キーで参照する側から順に作り直す
for t in ["repair_requests", "sales_monthly", "production_lots", "products"]:
    spark.sql(f"DROP TABLE IF EXISTS `{catalog}`.`{schema_name}`.`{t}`")

for t in ["products", "production_lots", "sales_monthly", "repair_requests"]:
    spec = TABLES[t]
    full = f"`{catalog}`.`{schema_name}`.`{t}`"
    df = (spark.read.format("csv")
          .option("header", True)
          .option("encoding", "UTF-8")
          .option("quote", '"').option("escape", '"')
          .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
          .option("mode", "FAILFAST")
          .schema(spec["schema"])
          .load(f"{raw_csv_path}/{t}.csv"))
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema_name}.{t}")
    spark.sql(f"COMMENT ON TABLE {full} IS '{spec['comment']}'")
    print(f"✅ {catalog}.{schema_name}.{t:<16} {spark.table(f'{catalog}.{schema_name}.{t}').count():>7,} 行")

## 4. 主キー・外部キーの設定と確認

主キー・外部キーは、データの値を検査しない**情報提供用の制約**です。Genie Agent がテーブルの結合のしかた（型番・ロット番号でつながること）を理解する手がかりになります。

In [0]:
def run(stmt):
    spark.sql(stmt.format(c=f"`{catalog}`.`{schema_name}`"))

# 主キーの列は NOT NULL にする必要がある
run("ALTER TABLE {c}.products ALTER COLUMN model_no SET NOT NULL")
run("ALTER TABLE {c}.production_lots ALTER COLUMN lot_no SET NOT NULL")
run("ALTER TABLE {c}.repair_requests ALTER COLUMN request_id SET NOT NULL")
for col_name in ["sales_month", "model_no", "region"]:
    run("ALTER TABLE {c}.sales_monthly ALTER COLUMN " + col_name + " SET NOT NULL")

run("ALTER TABLE {c}.products ADD CONSTRAINT pk_products PRIMARY KEY (model_no)")
run("ALTER TABLE {c}.production_lots ADD CONSTRAINT pk_production_lots PRIMARY KEY (lot_no)")
run("ALTER TABLE {c}.sales_monthly ADD CONSTRAINT pk_sales_monthly PRIMARY KEY (sales_month, model_no, region)")
run("ALTER TABLE {c}.repair_requests ADD CONSTRAINT pk_repair_requests PRIMARY KEY (request_id)")

run("ALTER TABLE {c}.production_lots ADD CONSTRAINT fk_lots_products FOREIGN KEY (model_no) REFERENCES {c}.products")
run("ALTER TABLE {c}.sales_monthly ADD CONSTRAINT fk_sales_products FOREIGN KEY (model_no) REFERENCES {c}.products")
run("ALTER TABLE {c}.repair_requests ADD CONSTRAINT fk_requests_products FOREIGN KEY (model_no) REFERENCES {c}.products")
run("ALTER TABLE {c}.repair_requests ADD CONSTRAINT fk_requests_lots FOREIGN KEY (lot_no) REFERENCES {c}.production_lots")
print("✅ 主キー・外部キーを設定しました")

In [0]:
expected = {"products": 10, "production_lots": 783, "sales_monthly": 1080, "repair_requests": 11884}
ok = True
for t, n in expected.items():
    actual = spark.table(f"{catalog}.{schema_name}.{t}").count()
    mark = "✅" if actual == n else "❌"
    ok &= actual == n
    print(f"{mark} {t:<16} {actual:>7,} 行（想定 {n:,} 行）")

n_docs = sum(len(dbutils.fs.ls(d.path)) for d in dbutils.fs.ls(docs_path))
print(f"{'✅' if n_docs == 34 else '❌'} {DOCS_VOLUME} Volume の文書 {n_docs} ファイル（想定 34 ファイル）")
assert ok and n_docs == 34, "件数が想定と異なります。CSV と文書のコピーを確認してください"
print("\n🎉 セットアップが完了しました。次は 01_シナリオと想定QA を開いてください")

In [0]:
display(spark.sql(f"""
SELECT request_id, received_at, model_no, lot_no, symptom_category, result_category, operator_memo
FROM `{catalog}`.`{schema_name}`.repair_requests
WHERE request_id BETWEEN 'R-2607-09140' AND 'R-2607-09150'
ORDER BY request_id
"""))

## 5. （任意）参加者への権限の付与

Genie Agent は、**質問した人の権限**でテーブルと Volume を読みます。共有する相手（グループ）に、次の権限を付けてください。講師がまとめて準備する場合は、`<グループ名>` を書き換えてコメントを外し、実行します。

| 対象 | 必要な権限 |
|---|---|
| カタログ | `USE CATALOG` |
| スキーマ | `USE SCHEMA` |
| テーブル4つ | `SELECT`（スキーマにまとめて付けてもよい） |
| `docs` Volume | `READ VOLUME` |
| SQL ウェアハウス | `CAN USE`（ウェアハウスの［権限］から付ける） |
| Genie Agent | `CAN RUN` 以上（Genie Agent の［共有］から付ける） |

In [0]:
# group = "<グループ名>"
# spark.sql(f"GRANT USE CATALOG ON CATALOG `{catalog}` TO `{group}`")
# spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA `{catalog}`.`{schema_name}` TO `{group}`")
# spark.sql(f"GRANT READ VOLUME ON VOLUME `{catalog}`.`{schema_name}`.`{DOCS_VOLUME}` TO `{group}`")